# North-coast campaign census

Every cell of the northern Spanish coast runs the same pipeline with
the same configuration and seeds — no per-site tuning. This notebook
reads whatever the campaign has produced so far (runs/*/marea/
result.json) and tabulates it: pixels with elevation, the largest
interior lag the correction actually used, and the hypsometric
integral per cell. It can be re-run at any stage of the campaign.

**You are here: 07.** The same pipeline, unchanged, over every cell of the northern Spanish coast.

```text
+- the evidence chain ------------------------------------------------+
|  datacube -> water masks -> per-pixel wet/dry series                |
|    01 what is estimable  ->  02 boundary audit  ->  03 operator vs  |
|    gauges  ->  04 elevations vs truth  ->  05 uncertainty and       |
|    hydraulic layers  ->  06 negatives kept  ->  07 coast census     |
+---------------------------------------------------------------------+
```

The campaign machinery in one picture:

```text
 north_coast_cells.json (130 terrain-following cells, from OSM coastline)
        |
        |   experiments/campaign_north_v2.py
        |   submit cube jobs -> harvest -> process (worker pool) -> purge
        v
 runs/<cell>/marea/result.json   --aggregated-->   results/
        (one verdict per cell)                     north_coast_census/
                                                   result.json  (read below)
```

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# run from the repo root so the results/ paths resolve
here = Path.cwd()
while not (here / "pyintertidal").is_dir():
    if here.parent == here:
        raise FileNotFoundError("repo root not found above " + str(Path.cwd()))
    here = here.parent
os.chdir(here)

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

In [2]:
import glob

# the census is one aggregated artifact; when the raw campaign outputs are
# on disk (runs/), rebuild it first so the notebook always shows the latest
CENSUS = "results/north_coast_census/result.json"
raw = sorted(glob.glob("runs/north_coast_cell*/marea/result.json"))
if raw:
    rows = []
    for p in raw:
        r = load(p)
        row = {"sitio": r.get("sitio", p), "estado": r.get("estado")}
        if r.get("estado") == "ok":
            taus = r.get("tau_usado_min") or []
            row.update({
                "n_px_cota": r.get("n_px_cota"),
                "tau_max_abs_min": (float(np.nanmax(np.abs(taus)))
                                    if taus else 0.0),
                "hipsometria_integral":
                    r.get("hipsometria", {}).get("integral"),
            })
        rows.append(row)
    json.dump({"nota": "aggregated from runs/*/marea/result.json",
               "n_cells": len(rows), "cells": rows},
              open(CENSUS, "w"), indent=1)

cells = load(CENSUS)["cells"]
print(f"cells processed so far: {len(cells)}")
print(f"{'cell':24s} {'state':10s} {'px':>8s} {'max tau':>9s} {'HI':>5s}")
for c in cells:
    ok = c["estado"] == "ok"
    px = str(c.get("n_px_cota")) if ok else "-"
    tau = "%+.0f min" % c["tau_max_abs_min"] if ok else "-"
    hi = ("%.2f" % c["hipsometria_integral"]
          if ok and c.get("hipsometria_integral") is not None else "-")
    print(f"{c['sitio']:24s} {c['estado']:10s} {px:>8s} {tau:>9s} {hi:>5s}")
ok_cells = [c for c in cells if c["estado"] == "ok"]
with_op = sum(1 for c in ok_cells if c.get("tau_max_abs_min", 0) > 0)
print(f"\ninterior tide detected (operator active) in "
      f"{with_op}/{len(ok_cells)} cells so far")

cells processed so far: 68
cell                     state            px   max tau    HI
north_coast_cell000      ok            51334   +23 min  0.53
north_coast_cell001      ok            24625   +13 min  0.54
north_coast_cell002      ok            29044   +98 min  0.49
north_coast_cell003      ok            11430   +29 min  0.48
north_coast_cell004      ok            11374   +26 min  0.46
north_coast_cell006      ok             3434   +34 min  0.44
north_coast_cell007      ok            26725   +28 min  0.58
north_coast_cell010      ok             3666    +0 min  0.42
north_coast_cell011      ok             9657   +24 min  0.37
north_coast_cell012      sin_intermareal        -         -     -
north_coast_cell014      sin_intermareal        -         -     -
north_coast_cell015      sin_intermareal        -         -     -
north_coast_cell016      sin_intermareal        -         -     -
north_coast_cell017      sin_intermareal        -         -     -
north_coast_cell018      sin_inte